CNN IN PYTORCH SU CIFAR-10: GESTIONE DEL COLORE E MONITORAGGIO AVANZATO

Segnale cromatrico, esploriamo come le macchine interpretano la profondità del colore.

CIFRA-10 è un dataset di immagini piccole, 32x32 a colori, divise in 10 classi (aereo, auto, uccello, gatto, cervo, cane, rana, cavallo, nave, camion)
Mentre MNIST è grigio, CIFAR-10 è a colori

Un'immagine in CIFRA-10 ha forma
(3x32x32)
(canali, altezza, larghezza)

Ogni immagine ha 3 'strati': Rosso, Verde, Blu. Un pixel non ha un solo colore, ma tre valori pixel=[R,G,B]
Esempio:
rosso=[255,0,0]
verde=[0,255,0]
blu=[0,0,255]
bianco=[255,255,255]
nero=[0,0,0]
Quando carichi CIFAR-10 con PyTorch, normalmente trasformi questi valori da 0-255 a valori circa tra 0 e 1 usando ToTensor()
Poi normalizzi ogni canale colore
transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),std=(0.2470, 0.2435, 0.2616))
Perchè il colore cambia tutto?

La Complessità del Colore
Il passaggio dai singoli canali alla profondità RGB
Mentre con MNIST abbiamo vissuto in un mondo bidimensionale, ogni pixel era solo un ombra grigio e ci ha permesso di studiare la geometria delle forme in bianco e nero, il dataset CIFAR-10 ci introduce alla relatà del segnale cromatico. Ogni pixel non è più un singolo valore, ma un vettore di tra intensità (valori): Rosso Verde Blu.
Questa transizione richiede che i nostri filtri convoluzionali non scorrano solo su una superficie piana, ma operino su un valume catturando le correlazioni tra i diversi canali colore per identificare pattern complessi.

Tensolri a Tre Canali
La struttura spaziale delle immagini CIFAR-10
Risoluzione e profondità: le immagini CIFAR10 sono matrici 32x32 con 3 canali. generano un volume in input di 3072 valori totali per ogni singola immagine.
Pytorch è pignolo sull'ordine delle dimensioni, mentri altri framework preferiscono avere i canali alla fine, PyTorch vuole i canali all'inizion
Batch + canali + altezza + larghezza.

Canali e Friltri. 
Poichè l'input ha profondità 3 anche i filtri del nostri primo leyer devono avere profondità 3. Questo significa triplicare i calcoli rispetto a MINST
Per gestire questo carico senza far esplodere i gradienti, useremo la normalizzazione per canale.,
Una volta normalizzati i colori, la rete inizia a vedere la semantica.

Gerarchia del Colore.
Dal pixel alla semantica cromatica.
Nelle fasi inziali, la rete può specializzarsi nel rilevare contrasti di colore (es. il giallo di un becco il blu dell'acquia). Questa capacità (es. il giallo di un becco contro il blu dell'acqua). Questa capacità è preclusa ai modelli che lavorano solo in scale di grigio.

In PyTorch il caricamento del dato non è un evento statico ma una pipeline dinamica. Utilizzeremo il modulo 'transforms' per preparare le immagini prima che queste vengano somministrate al modello.
Questa fase è cruciale per la generalizzazione: una corretta trasformazione impedisce alla rete di memorizzare pixel rumorosi e favorisce l'apprendimento di feature robuste.
Se prepariamo bene i dati qui, la rete sarà molto più brava a riconoscere un oggetto anche con una foto sgrananta.

Standarizzazione del Flusso
Converitre immagini in tensori pronti al trainig
- ToTensor: è il nostro traduttore, trasforma l'immagine PIL o array NumPy (leggibile dall'uomo) in un tensore di float e scalla i valori nell'intervallo 0,1
- Z-Score Normalization: sottraendo la media e dividendo per la deviazione stanrda, rendiamo il gradiente più stabile e veloce, centrando i dati attorno allo zero.
Composizione: transforms.Compose permette di concatenare 
L'operazione di normalizzazoine trasforma ogni canale sottranendo la media specifica e dividendo per lo scarto quadratico medio.
Ma la pipelne può fare molto di più di una semplice pulizia.

Efficienza della Pipeline
Possiamo rendere la nostra rete pià furba aggiungendo piccoli trucchi.
Data Augmentation di Base: Sebbene l'augmentation verrà trattata in seguito, piccoli accorgimenti come il 'RandomHorizontalFlip' possono già migliorare la robustezza del modello.
Gestione Batch: Il DataLoader organizza le trasformazioni in parallelo usando diversi 'worker', ottimizzando i tempi di attesa tra un'epoca e l'altra.
Mapping delle Classi: CIFAR-10 utilizza indici interi, è compito della pipeline mappare correttamente questi indici ai nomi delle classi leggibili per l'utente finale.

Il Ciclo di Caricamento
Dal disco alla GPU
A differenza di Keras dove i dati sono caricati tutti in memoria RAM, PyTorch usa gli iteratori. Questo approccio permette di gestire dataset che superano la capacità della RAM fisica del sistema.
La trasformazione avviene 'on-the-fly' durante l'iterazione (solo quando servono), garantendo che il processore grafico riceva sempre dati pronti per il calcolo dei gradienti.
Ora che la rete sta imparando come facciamo a sapere se sta capendo tutto?

Diagnostica e Monitoraggio
Analisi fine dell'apprendimento per categoria
Un errore comune è guardare solo l'accuratezza globale. Tuttavia, un modello potrebbe essere eccellente nel riconoscere navi ma fallire miseramente con i gatti, portando a una media ingannevole.
Ci vuole un sistema di monitoraggio che traccia le performance specifica per ogni classe, permettendoci di identificare eventuali confusioni semantiche del modello.

Loss per Classe
Identificare i punti deboli del classificatore.
Contare successi e fallimenti per ogni singola etichetta
- Confusion Matrix Implicita: monitorare quali classi vengono scambiate più frequentemente aiuta a capire se la rete manca di dettagli discriminanti.
- Valutazione per Classe: calcolando il rapporto tra predizoini correte e totali per ogni etichetta, otteniamo una vista granulare della salute del modello.
- Accumulatori di Statistiche: durante il test, memorizzando separatamente i risuotati per le 10 classi di CIFAR10 per generare un report finale di precisione
- L'accuretezza per la classe i-esima è il rapporto tra i veri positivi e il totale degli esempi appartenenti a quella categoria.

Poi per correggere questi errori dobbiamo utilizzare funzioni di costo giuste.

Techinche di Monitoraggio
Utilizzeremo la nn.CrossEntropyLoss che combina internamente LogSoftmax e NLLLoss, ideale per problemi multiclasse come CIFAR10
Possiamo utilizzare l'ottimizzatorei SGD con momentum per navigare il paesaggio della loss più accidentato rispetto a quello di MNIST (oppure un Adam più veloce che agisce come una pallina che rotola giù da un colle).
La perdita misurata sul set di validazione alla fine di ogni epoca, permette di prevenire l'overfitting e decidere quando interrompere l'addestramento.

Interpretazione degli errori.
Perchè la rete confonde cani e gatti?
Il report per classe permette di scoprire oggetti con strutture simili (come gatti e cani) e che tendono ad avere accuratezze inferiore rispetto a classi molto diverse (come navi e aerei)
Questa analisi ci giuderà nella futura implementazione di tecniche di regolarizzazione più agressive o nell'aumento della profondità dei layer convoluzionali.


In [ ]:
import torch
import torch.nn as nn
import torch.optim
from torch.utils.data import DataLoader
import torchvision
from torchvision.transforms import v2
import matplotlib.pyplot as plt
import numpy as np

# --- 1. CONFIGURAZONE HARDWARE (Standard 2025)
# Selezioniamo il miglior backend disponibile CUDA (NVIDIA), MPS (Apple Sillicon) o CPU (default)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")   
print(f"Training on device: {device}")

# --- 2. DATA PIPELINE (Transform v2)
# Le v2 torchvision sono ottimizzate per i tensori e più veloci
transform = v2.Compose([
    v2.ToImage(),  # converte le immagini in un oggetto tensore immagine
    v2.ToDtype(torch.float32,scale=True),  # Normalizza i pixel da(0,255) a (0,1) (float32)
    # Normalizzazione con media e deviazione standard a 0.5 per i 3 canali (RGB)
    # Questo sposta i dati nel range (-1,1) aiutando la stabilità mamematica del gradienti
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # spostiamo in un range tra -1 e 1
])

# Caricamento Dataset CIFAR-10 (immagini 32x32 a colori), 10 classi
trainset=torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

# Test set: fondamentale per valutare la capacità di generalizzazione (immagini mai viste)
testset=torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

classes=("aereo","auto","uccello","gatto","cervo","cane","rana","cavallo","nave","camion")

# --- 3. ARCHITETTURA DELLA RETE (Modern CNN)
class ModernCNN(nn.Module):
    def __init__(self):

            # FEATURE EXTRACTOR: La parte 'visiva' cheim para a riconoscere forme e colori
        self.feature_exstractor=nn.Sequential(
            # Conv1: 3 canali in (RGB), 32 filtri (mappa d uscita)
            # Padding=1 mantiene la dimensione spaziale (32x32) dopo la convoluzione
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # Input: 3x32x32, Output: 32x32x32
            nn.BatchNorm2d(32),  # Stabilizza l'apprendimento normalizzando i pesi durante il training
            nn.ReLU(),  # Funzione di attivazione non lineare, permette di imparare pattern complessi
            nn.MaxPool2d(2,2), # Dimezza le dimensioni spaziali (da 32x32 a 16x16)

            # Conv2:  riceve 32 canali, produce 64 mappe più astratte
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2) # Riduce ulteriormente da 16x16 a 8x8
        )

        # CLASSIFIER: La parte 'logica' che prende le feature e decide la classe
        self.classifier=nn.Sequential(
            nn.Flatten(), # Trasforma il cubo 8x8x64 in un vettore piatto da 4096 elementi
            nn.Linear(64*8*8,125), # Primo strato denso (Fully Connected)
            nn.ReLU(),
            nn.Dropout(0.2), # Spegne il 20% dei neuroni casualmente per evitare l overfitting
            nn.Linear(128,10) # Strato finale: 10 classi (uno score per ogni classe)
        )
    def forward (self,x):
        #Flusso dei dati: Immagine -> Feature -> Classi
        x=self.feature_exstractor(x)
        return self.classifier(x)
    
# Istanziamo il modello e lo spostiamo sulla memoria del device (GPU/MPS/CPU)
model=ModernCNN.to(device)

# Funzione di perdita: Categorical Cross-Entorpy (standard per classificazione multi-classe)
criterio=nn.CrossEntropyLoss()

# Ottimizzatore: Adam (Adam con Weight dacay migliorato). Lo standard nel 2025 per velocità e stabilità
optimizer=optim.Adam(model.parameters(),lr=0.001)

# --- 4. LOOP DI ADDESTRAMENTO
epochs=3
print("\n --- INIZIO ADDESTRAMENTO ---")
for epoch in range(epochs):
        model.train() # Imposta il modello in modalità addestramento (attiva Dropout e BatchNorms)
        running_loss, correct, totale=0.0,0,0

        for i, (image, labels) in enumerate(trainloader):
              # Spostiamo i dati sul device (GPU o CPU)
              image, labels=images.to(device),label.to(device)
            
    




        super(ModernCNN, self).__init__()
        # Blocchi convoluzionali con BatchNorm e ReLU
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # Input: 3x32x32, Output: 32x32x32
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)  # Output: 32x16x16
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # Output: 64x16x16
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)  # Output: 64x8x8
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),  # Output: 128x8x8
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)  # Output: 128x4x4
        )
        self.fc = nn.Linear(128 * 4 * 4, 10)  # Fully connected layer for classification

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = self.fc(x)
        return x
    

Training on device: cpu
